# FFC Sona Urea: forecasting analysis notebook

*Company-Calibrated Synthetic Dataset for Academic Analysis — illustrative figures, calibrated to Fauji Fertilizer Company Limited's disclosed scale, not actual company records.*

Run all cells in order (Runtime → Run all). Error convention: Error = Actual − Forecast.

In [ ]:
# =====================================================================
# FFC Sona Urea — 12-week holdout comparison of forecasting methods
# Company-Calibrated Synthetic Dataset for Academic Analysis — illustrative
# figures, calibrated to Fauji Fertilizer Company Limited's disclosed scale,
# not actual company records.
# =====================================================================

## Setup and data
Upload `FFC_Sona_Urea_Synthetic_Weekly_v2_FROZEN.csv` when prompted.

In [ ]:
# %% [1] Setup and data -------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LABEL = ("Company-Calibrated Synthetic Dataset for Academic Analysis — illustrative figures, "
         "calibrated to Fauji Fertilizer Company Limited's disclosed scale, not actual company records.")
CSV_NAME = "FFC_Sona_Urea_Synthetic_Weekly_v2_FROZEN.csv"
HOLDOUT = 12

try:                                   # In Colab: upload the frozen CSV when prompted
    from google.colab import files
    import os
    if not os.path.exists(CSV_NAME):
        files.upload()
except ImportError:
    pass                               # Running outside Colab: CSV must be in the working folder

df = pd.read_csv(CSV_NAME, parse_dates=["Week_Start_Date"]).sort_values("Week_Start_Date").reset_index(drop=True)

# Skip this split if you already have train_df / test_df
train_df = df.iloc[:-HOLDOUT].copy()
test_df  = df.iloc[-HOLDOUT:].copy()

# Leakage guard: test must come strictly after train, and models only ever see train_df
assert train_df["Week_Start_Date"].max() < test_df["Week_Start_Date"].min()
y_train = train_df["Actual_Demand"].to_numpy(dtype=float)
y_test  = test_df["Actual_Demand"].to_numpy(dtype=float)
H = len(test_df)
print(f"Train: {train_df.Week_Start_Date.min().date()} to {train_df.Week_Start_Date.max().date()} ({len(train_df)} weeks)")
print(f"Test : {test_df.Week_Start_Date.min().date()} to {test_df.Week_Start_Date.max().date()} ({H} weeks)")

# All forecasts below are made ONCE, from the last training week, for all 12 test weeks
# (fixed-origin, 1- to 12-step-ahead). Nothing is updated with test actuals.
forecasts = {}

## Section 8: Forecast models (fitted on 144 training weeks only)

In [ ]:
# %% [2] Naive and Seasonal Naive ---------------------------------------
forecasts["Naive"] = np.repeat(y_train[-1], H)                       # last observed week, held flat
season = 52
forecasts["Seasonal Naive"] = y_train[len(y_train) - season : len(y_train) - season + H]  # same week last year (all in train)

In [ ]:
# %% [3] Simple Moving Average (4-week) ---------------------------------
forecasts["Moving Average (4 wk)"] = np.repeat(y_train[-4:].mean(), H)

In [ ]:
# %% [4] Simple Exponential Smoothing (alpha 0.3 and 0.6) ---------------
def ses_level(series, alpha):
    level = series[0]
    for value in series[1:]:
        level = alpha * value + (1 - alpha) * level
    return level

forecasts["SES (alpha 0.3)"] = np.repeat(ses_level(y_train, 0.3), H)
forecasts["SES (alpha 0.6)"] = np.repeat(ses_level(y_train, 0.6), H)

In [ ]:
# %% [5] Seasonal decomposition + Event_Flag + smoothed level -----------
# Step a: put all weeks on the same business scope. Port Qasim urea joins FFC from Jul 2024
#         (Merger_Flag = 1). Scope factor from the 2025 annual report: 2,886 / (2,886 - 449) = 1.184.
MERGER_SCOPE = 2886 / (2886 - 449)
if "Merger_Flag" in train_df.columns:
    scope_train = np.where(train_df["Merger_Flag"] == 1, 1.0, MERGER_SCOPE)
    scope_test  = np.where(test_df["Merger_Flag"] == 1, 1.0, 1 / MERGER_SCOPE)
else:
    scope_train = np.ones(len(train_df)); scope_test = np.ones(H)
y_scoped = y_train * scope_train

# Step b: estimate seasonality (12 monthly indices) and the Event_Flag uplift together,
#         on TRAIN ONLY, as log(demand / 52-week centred moving average) ~ month + Event_Flag
month_train = (train_df["Week_Start_Date"] + pd.Timedelta(days=3)).dt.month.to_numpy()
month_test  = (test_df["Week_Start_Date"]  + pd.Timedelta(days=3)).dt.month.to_numpy()
cma = pd.Series(y_scoped).rolling(52, center=True).mean().rolling(2).mean().shift(-1).to_numpy()
ok = ~np.isnan(cma)
X = np.column_stack([(month_train == m).astype(float) for m in range(1, 13)] + [train_df["Event_Flag"].to_numpy(float)])
coef, *_ = np.linalg.lstsq(X[ok], np.log(y_scoped[ok] / cma[ok]), rcond=None)
seasonal_index = np.exp(coef[:12]); seasonal_index /= np.exp(np.log(seasonal_index).mean())
event_factor = np.exp(coef[12])
print("\nMonthly seasonal index:", dict(zip(range(1, 13), seasonal_index.round(2))))
print(f"Event_Flag uplift on top of month: x{event_factor:.3f}")

# Step c: remove seasonality and event effect, then smooth the remaining level.
#         Alpha chosen by one-step-ahead error on TRAIN ONLY.
fac_train = seasonal_index[month_train - 1] * np.where(train_df["Event_Flag"] == 1, event_factor, 1.0)
z = y_scoped / fac_train
best = None
for a in np.arange(0.05, 0.96, 0.05):
    level, sse = z[0], 0.0
    for v in z[1:]:
        sse += (v - level) ** 2
        level = a * v + (1 - a) * level
    if best is None or sse < best[0]:
        best = (sse, a, level)
_, alpha_star, final_level = best
print(f"Level smoothing alpha (fit on train): {alpha_star:.2f}")

# Step d: forecast = level x month index x event uplift (Event_Flag is a calendar known in advance)
fac_test = seasonal_index[month_test - 1] * np.where(test_df["Event_Flag"] == 1, event_factor, 1.0)
forecasts["Seasonal Decomp + Event"] = final_level * fac_test * scope_test

## Section 8: Model comparison on the 12-week holdout

In [ ]:
# %% [6] Model comparison table -----------------------------------------
def metrics(actual, forecast):
    err = actual - forecast                      # Error = Actual - Forecast (+ = under-forecast)
    mad = np.abs(err).mean()
    return {"MAD": mad,
            "MAPE %": np.mean(np.abs(err) / actual) * 100,
            "RMSE": np.sqrt(np.mean(err ** 2)),
            "WAPE %": np.abs(err).sum() / actual.sum() * 100,
            "Bias": err.mean(),
            "Tracking Signal": err.sum() / mad}

table = pd.DataFrame({name: metrics(y_test, f) for name, f in forecasts.items()}).T
table = table.sort_values("WAPE %")
pd.options.display.float_format = "{:,.1f}".format
print("\n" + LABEL)
print("12-week holdout results (tonnes; Error = Actual - Forecast)")
try:
    display(table)          # Colab / Jupyter
except NameError:
    print(table.to_string())

In [ ]:
# %% [7] Plot all forecasts against actual test values ------------------
fig, ax = plt.subplots(figsize=(13, 6))
ctx = train_df.tail(26)
ax.plot(ctx["Week_Start_Date"], ctx["Actual_Demand"] / 1000, color="grey", lw=1.5, label="Actual (last 26 training weeks)")
ax.plot(test_df["Week_Start_Date"], y_test / 1000, color="black", lw=3, marker="o", label="Actual (holdout)")
styles = {"Naive": ("#9E9E9E", ":"), "Seasonal Naive": ("#E07B00", "--"), "Moving Average (4 wk)": ("#8E44AD", ":"),
          "SES (alpha 0.3)": ("#2E86C1", "-."), "SES (alpha 0.6)": ("#16A085", "-."), "Seasonal Decomp + Event": ("#C0392B", "-")}
for name, f in forecasts.items():
    c, ls = styles.get(name, (None, "-"))
    ax.plot(test_df["Week_Start_Date"], f / 1000, color=c, ls=ls, lw=2.2 if "Decomp" in name else 1.8,
            label=f"{name} (WAPE {table.loc[name, 'WAPE %']:.1f}%)")
ax.axvline(test_df["Week_Start_Date"].min() - pd.Timedelta(days=3), color="black", lw=0.8)
ax.text(test_df["Week_Start_Date"].min(), ax.get_ylim()[1] * 0.97, "  holdout starts", fontsize=9, va="top")
ax.set_ylabel("Thousand tonnes per week")
ax.set_title("FFC Sona Urea — 12-week holdout: forecasts vs actual", loc="left", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=8.5, loc="upper left", ncol=2)
fig.text(0.01, 0.005, LABEL, fontsize=7.5, color="#C00000", style="italic")
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig("FFC_holdout_forecast_comparison.png", dpi=180)
plt.show()

## Section 9: Holdout validation and leakage checks

In [ ]:
# %% [8] SECTION 9 — Holdout validation of the selected model ------------
# Selected model: Seasonal Decomp + Event. Wrap the whole fit in ONE function that receives
# only a training frame, so we can prove the forecast never touches holdout actuals.
def fit_decomp_event(train, future):
    """Fit on `train` only; return (holdout forecast, in-sample one-step forecasts, details)."""
    y = train["Actual_Demand"].to_numpy(dtype=float)
    sc_tr = np.where(train["Merger_Flag"] == 1, 1.0, MERGER_SCOPE) if "Merger_Flag" in train else np.ones(len(train))
    sc_fu = np.where(future["Merger_Flag"] == 1, 1.0, 1 / MERGER_SCOPE) if "Merger_Flag" in future else np.ones(len(future))
    ys = y * sc_tr
    m_tr = (train["Week_Start_Date"] + pd.Timedelta(days=3)).dt.month.to_numpy()
    m_fu = (future["Week_Start_Date"] + pd.Timedelta(days=3)).dt.month.to_numpy()
    cma_ = pd.Series(ys).rolling(52, center=True).mean().rolling(2).mean().shift(-1).to_numpy()
    ok_ = ~np.isnan(cma_)
    X_ = np.column_stack([(m_tr == m).astype(float) for m in range(1, 13)] + [train["Event_Flag"].to_numpy(float)])
    c_, *_ = np.linalg.lstsq(X_[ok_], np.log(ys[ok_] / cma_[ok_]), rcond=None)
    si_ = np.exp(c_[:12]); si_ /= np.exp(np.log(si_).mean()); ef_ = np.exp(c_[12])
    f_tr = si_[m_tr - 1] * np.where(train["Event_Flag"] == 1, ef_, 1.0)
    z_ = ys / f_tr
    best_ = None
    for a in np.arange(0.05, 0.96, 0.05):
        lv, sse, fitted = z_[0], 0.0, [z_[0]]
        for v in z_[1:]:
            sse += (v - lv) ** 2
            lv = a * v + (1 - a) * lv
            fitted.append(lv)
        if best_ is None or sse < best_[0]:
            best_ = (sse, a, lv, np.array(fitted))
    _, a_, lv_, lv_path = best_
    one_step = np.r_[np.nan, lv_path[:-1]] * f_tr / sc_tr          # in-sample one-step-ahead fit
    f_fu = si_[m_fu - 1] * np.where(future["Event_Flag"] == 1, ef_, 1.0)
    return lv_ * f_fu * sc_fu, one_step, {"alpha": a_, "event_factor": ef_}

fc_sel, fit_sel, info = fit_decomp_event(train_df, test_df)
assert np.allclose(fc_sel, forecasts["Seasonal Decomp + Event"])   # same model as the comparison table

val = test_df[["Week_Start_Date", "Actual_Demand"]].copy()
val["Forecast"] = fc_sel.round(0)
val["Error (A-F)"] = val["Actual_Demand"] - val["Forecast"]
val["APE %"] = (val["Error (A-F)"].abs() / val["Actual_Demand"] * 100).round(1)
for col in ["Promo_Flag", "Promo_Depth", "Event_Flag", "Stockout_Flag", "Exceptional_Event"]:
    if col in test_df: val[col] = test_df[col].values
wape_sel = val["Error (A-F)"].abs().sum() / val["Actual_Demand"].sum() * 100
bias_sel = val["Error (A-F)"].mean()

In [ ]:
# %% [9] Plot: full history with train vs holdout marked, plus holdout close-up
top5 = val.reindex(val["Error (A-F)"].abs().sort_values(ascending=False).index).head(5)
t0, t1 = test_df["Week_Start_Date"].min(), test_df["Week_Start_Date"].max() + pd.Timedelta(days=7)
fig, (ax, bx) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [1, 1.15]})
# Top: whole series
ax.axvspan(train_df["Week_Start_Date"].min(), t0, color="#EAF0F8", zorder=0)
ax.axvspan(t0, t1, color="#FADBD8", zorder=0)
ax.plot(train_df["Week_Start_Date"], y_train / 1000, color="#1F3864", lw=1.3, label="Actual — train")
ax.plot(train_df["Week_Start_Date"], fit_sel / 1000, color="#7FB3D5", lw=1, ls="--", label="Model fit — train (one-step-ahead)")
ax.plot(test_df["Week_Start_Date"], y_test / 1000, color="black", lw=2.2, label="Actual — holdout")
ax.plot(test_df["Week_Start_Date"], fc_sel / 1000, color="#C0392B", lw=2.2, label="Forecast — holdout")
ax.text(train_df["Week_Start_Date"].min() + pd.Timedelta(days=20), 110,
        f"TRAIN PERIOD: {len(train_df)} weeks, {train_df.Week_Start_Date.min():%d %b %Y} to {train_df.Week_Start_Date.max():%d %b %Y}",
        fontsize=10, fontweight="bold", color="#1F3864")
ax.annotate(f"HOLDOUT: {H} weeks\n{t0:%d %b} to {test_df.Week_Start_Date.max():%d %b %Y}", xy=(t0, 30), xytext=(t0 - pd.Timedelta(days=150), 27),
            fontsize=10, fontweight="bold", color="#C0392B", arrowprops=dict(arrowstyle="->", color="#C0392B"))
ax.set_ylim(20, 118); ax.set_ylabel("Thousand tonnes / week"); ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=8.5, loc="upper right", ncol=2)
ax.set_title("Holdout validation — Seasonal Decomp + Event: train → forecast holdout → reveal actual → measure → diagnose",
             loc="left", fontweight="bold", fontsize=11)
# Bottom: holdout close-up
bx.set_facecolor("#FEF5F4")
bx.plot(test_df["Week_Start_Date"], y_test / 1000, color="black", lw=2.6, marker="o", label="Actual (revealed after forecasting)")
bx.plot(test_df["Week_Start_Date"], fc_sel / 1000, color="#C0392B", lw=2.6, marker="s", label="Forecast (made once, from 29 Sep 2025)")
for d_, a_, f_ in zip(test_df["Week_Start_Date"], y_test / 1000, fc_sel / 1000):
    bx.vlines(d_, min(a_, f_), max(a_, f_), color="#7F8C8D", lw=1, ls=":")
for rank, (_, r) in enumerate(top5.iterrows(), 1):
    up = r["Error (A-F)"] > 0
    bx.annotate(f"#{rank}  {r['Error (A-F)']/1000:+.1f} KT", (r["Week_Start_Date"], r["Actual_Demand"] / 1000),
                xytext=(0, 14 if up else -20), textcoords="offset points", ha="center", fontsize=9, fontweight="bold", color="#C0392B")
bx.set_ylim(25, 118); bx.set_ylabel("Thousand tonnes / week"); bx.grid(axis="y", alpha=0.3)
bx.set_title(f"Holdout close-up: forecast vs actual, five largest misses ranked (WAPE {wape_sel:.1f}%, bias {bias_sel/1000:+.1f} KT/wk)", loc="left", fontsize=10.5, fontweight="bold")
bx.legend(fontsize=9, loc="upper left")
fig.text(0.01, 0.005, LABEL, fontsize=7.5, color="#C00000", style="italic")
plt.tight_layout(rect=(0, 0.02, 1, 1)); plt.savefig("FFC_holdout_validation.png", dpi=180); plt.show()

In [ ]:
# %% [10] Five largest misses --------------------------------------------
print("Five largest misses (Error = Actual - Forecast; + = under-forecast)")
try:
    display(top5)
except NameError:
    print(top5.to_string())

In [ ]:
# %% [11] Leakage and plausibility checks --------------------------------
checks = {}
checks["1. Train ends before holdout starts"] = train_df["Week_Start_Date"].max() < test_df["Week_Start_Date"].min()
# 2. Scramble the holdout actuals: if the forecast changes, holdout data leaked into the fit
scrambled = test_df.copy()
scrambled["Actual_Demand"] = np.random.default_rng(0).integers(1, 10**6, H)
fc_scr, _, _ = fit_decomp_event(train_df, scrambled)
checks["2. Forecast unchanged when holdout actuals are scrambled"] = np.allclose(fc_scr, fc_sel)
# 3. Forecast unchanged if holdout demand columns are deleted entirely
fc_drop, _, _ = fit_decomp_event(train_df, test_df.drop(columns=["Actual_Demand"]))
checks["3. Forecast unchanged when holdout Actual_Demand is deleted"] = np.allclose(fc_drop, fc_sel)
# 4. Only calendar fields are read from the holdout rows
# 4. Forecast can be built from calendar fields alone (all known in advance)
cal_cols = [c for c in ["Week_Start_Date", "Event_Flag", "Merger_Flag"] if c in test_df]
fc_cal, _, _ = fit_decomp_event(train_df, test_df[cal_cols])
checks["4. Forecast needs only calendar fields from holdout rows: " + ", ".join(cal_cols)] = np.allclose(fc_cal, fc_sel)
# 5. Plausibility: holdout accuracy should not beat in-sample accuracy by a wide margin
e_in = (y_train - fit_sel)[~np.isnan(fit_sel)]
wape_in = np.abs(e_in).sum() / y_train[~np.isnan(fit_sel)].sum() * 100
wape_out = np.abs(y_test - fc_sel).sum() / y_test.sum() * 100
checks[f"5. Not suspiciously perfect: holdout WAPE {wape_out:.1f}% vs in-sample {wape_in:.1f}%"] = wape_out > 0.5 * wape_in
checks["6. Errors have both signs (not a copy of actuals)"] = (val["Error (A-F)"] > 0).any() and (val["Error (A-F)"] < 0).any()
for k, v in checks.items():
    print(("PASS  " if v else "FAIL  ") + k)

## Section 10: Regression and causal drivers
Model A is the required specification; Model B adds Merger_Flag. Effects are associations, not causes.

In [ ]:
# %% [12] SECTION 10 — Model A: the required regression, fitted on train_df only
import statsmodels.formula.api as smf
print(LABEL)
model_A = smf.ols("Actual_Demand ~ Price + Promo_Flag + Event_Flag + Economic_Index", data=train_df).fit()
print(model_A.summary())

In [ ]:
# %% [13] Model B: adds Merger_Flag to control the main confounder; Newey-West errors for autocorrelation
model_B = smf.ols("Actual_Demand ~ Price + Promo_Flag + Event_Flag + Economic_Index + Merger_Flag", data=train_df).fit()
print(model_B.summary())
model_B_nw = model_B.get_robustcov_results(cov_type="HAC", maxlags=4)
print("Newey-West p-values:", dict(zip(model_B.params.index, model_B_nw.pvalues.round(4))))

In [ ]:
# %% [14] Side-by-side table and plain-English coefficient sentences
def tidy(m, name):
    ci = m.conf_int()
    t = pd.DataFrame({"Coefficient": m.params, "p-value": m.pvalues, "95% CI low": ci[0], "95% CI high": ci[1]})
    t.columns = pd.MultiIndex.from_product([[name], t.columns])
    return t
comparison = pd.concat([tidy(model_A, "Model A (required)"), tidy(model_B, "Model B (+ Merger_Flag)")], axis=1)
pd.options.display.float_format = "{:,.4f}".format
try:
    display(comparison)
except NameError:
    print(comparison.to_string())
for name, m in [("Model A", model_A), ("Model B", model_B)]:
    print(f"{name}: R-squared = {m.rsquared:.3f}, Adjusted R-squared = {m.rsquared_adj:.3f}, n = {int(m.nobs)}")
units = {"Price": "PKR 1 increase in list price per bag", "Promo_Flag": "week with a dealer discount scheme (vs no scheme)",
         "Event_Flag": "week inside a Rabi/Kharif peak window (vs outside)", "Economic_Index": "1-point rise in the farm-economics index",
         "Merger_Flag": "week after Port Qasim urea joined (Jul 2024 onward)"}
print("\nPlain-English reading of Model B (holding the other variables constant; associations, not causes):")
for var in model_B.params.index[1:]:
    print(f"- Each {units[var]} is associated with {model_B.params[var]:+,.0f} tonnes/week of demand (p = {model_B.pvalues[var]:.3f}).")
# Price stability check (audit): the coefficient changes sign across specifications, so no elasticity is claimed
month = (train_df["Week_Start_Date"] + pd.Timedelta(days=3)).dt.month.astype(str)
specs = {"Model B": model_B,
         "Model B + month dummies": smf.ols("Actual_Demand ~ Price + Promo_Flag + Event_Flag + Economic_Index + Merger_Flag + C(month)", data=train_df.assign(month=month)).fit(),
         "Pre-merger weeks only": smf.ols("Actual_Demand ~ Price + Promo_Flag + Event_Flag + Economic_Index", data=train_df[train_df.Merger_Flag == 0]).fit(),
         "Post-merger weeks only": smf.ols("Actual_Demand ~ Price + Promo_Flag + Event_Flag + Economic_Index", data=train_df[train_df.Merger_Flag == 1]).fit()}
print("\nPrice coefficient by specification (t per PKR):", {k: round(v.params["Price"], 2) for k, v in specs.items()})

In [ ]:
# %% [15] Multicollinearity (VIF) and residual autocorrelation (Durbin-Watson)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
X = model_B.model.exog
vif = pd.Series([variance_inflation_factor(X, i) for i in range(1, X.shape[1])], index=model_B.model.exog_names[1:], name="VIF")
print(vif.round(2))
print("Durbin-Watson:", round(durbin_watson(model_B.resid), 2))

## Section 11: Stress test of the 2026 supply decision

In [ ]:
# %% [16] SECTION 11 — Stress test the 2026 supply decision
# ---------- Assumptions (Real = FFC Annual Report 2025; Assumed = stated judgement) ----------
FULL_RATE   = 790_000 / 13      # t/week at full rate = Q2-2025 urea production 790 KT / 13 weeks (Real)
TA_LOSS     = 95_000            # t lost in one plant turnaround = Q4-2025 shortfall when Goth Machhi II was down (Real-derived)
TA_WEEKS    = 7
PQ_RATE     = 456_000 / 52      # Port Qasim urea, t/week (Real: 456 KT produced in 2025)
OPENING     = 150_000           # urea stock on 1 Jan 2026, t (Assumed: report gives no tonnage, says "lowest in industry")
PRICE_T     = 4_020 * 20        # PKR per tonne (list price per 50 kg bag x 20)
GM          = 130.6 / 432.4     # gross margin 2025 (Real)
COST_T      = PRICE_T * (1 - GM)
FIN_RATE    = 0.105             # SBP policy rate, Dec 2025 (Real) — financing cost only
DISC_T      = 1_250 * 20        # PKR discount cost per extra tonne moved by schemes (Section 10 regression)
PROMO_LIFT  = model_B.params["Promo_Flag"] / train_df["Actual_Demand"].mean()   # ~0.17 from Section 10

# ---------- Planning baseline: selected model refit on ALL 156 weeks, scheme lift removed ----------
def fit_baseline(hist, future, promo_lift):
    y = hist["Actual_Demand"].to_numpy(float)
    ys = y * np.where(hist["Merger_Flag"] == 1, 1.0, MERGER_SCOPE) / np.where(hist["Promo_Flag"] == 1, 1 + promo_lift, 1.0)
    m_h = (hist["Week_Start_Date"] + pd.Timedelta(days=3)).dt.month.to_numpy()
    m_f = (future["Week_Start_Date"] + pd.Timedelta(days=3)).dt.month.to_numpy()
    cma_ = pd.Series(ys).rolling(52, center=True).mean().rolling(2).mean().shift(-1).to_numpy(); ok_ = ~np.isnan(cma_)
    X_ = np.column_stack([(m_h == m).astype(float) for m in range(1, 13)] + [hist["Event_Flag"].to_numpy(float)])
    c_, *_ = np.linalg.lstsq(X_[ok_], np.log(ys[ok_] / cma_[ok_]), rcond=None)
    si_ = np.exp(c_[:12]); si_ /= np.exp(np.log(si_).mean()); ef_ = np.exp(c_[12])
    z_ = ys / (si_[m_h - 1] * np.where(hist["Event_Flag"] == 1, ef_, 1.0))
    best_ = None
    for a in np.arange(0.05, 0.96, 0.05):
        lv, sse = z_[0], 0.0
        for v in z_[1:]:
            sse += (v - lv) ** 2; lv = a * v + (1 - a) * lv
        if best_ is None or sse < best_[0]: best_ = (sse, a, lv)
    return best_[2] * si_[m_f - 1] * np.where(future["Event_Flag"] == 1, ef_, 1.0)

plan = pd.DataFrame({"Week_Start_Date": pd.date_range("2025-12-29", periods=52, freq="7D")})
thu_ = plan["Week_Start_Date"] + pd.Timedelta(days=3); md_ = thu_.dt.month * 100 + thu_.dt.day
plan["Event_Flag"] = (((md_ >= 1115) | (md_ <= 131)) | ((md_ >= 701) & (md_ <= 815))).astype(int)
plan["Merger_Flag"] = 1
plan["Quarter"] = thu_.dt.quarter
base = fit_baseline(df, plan, PROMO_LIFT)
print(LABEL)
print("2026 baseline demand by quarter (KT):", (pd.Series(base).groupby(plan["Quarter"]).sum() / 1000).round(0).to_dict(),
      f"| year: {base.sum()/1000:,.0f} KT")

# Safety floor = 1.65 x std dev of 4-week demand error, from rolling-origin backtests on history
blk = []
for o in range(78, len(df) - 4, 2):
    tr_, te_ = df.iloc[:o], df.iloc[o:o + 4]
    f_ = fit_baseline(tr_, te_, PROMO_LIFT) * np.where(te_["Promo_Flag"] == 1, 1 + PROMO_LIFT, 1.0)
    blk.append(te_["Actual_Demand"].sum() - f_.sum())
FLOOR = round(1.65 * np.std(blk), -3)
print(f"Safety stock floor: {FLOOR/1000:.0f} KT (1.65 x std dev of 4-week forecast error)")

# ---------- Supply plan and weekly stock simulation ----------
def week_idx(date): return int(np.where(plan["Week_Start_Date"] == pd.Timestamp(date))[0][0])
def production(ta_start="2026-02-23", cuts=()):
    p = np.full(52, FULL_RATE)
    if ta_start: i = week_idx(ta_start); p[i:i + TA_WEEKS] -= TA_LOSS / TA_WEEKS
    for start, n, loss in cuts:
        j = week_idx(start); p[j:j + n] -= loss
    return p
def simulate(demand, prod):
    s, path, lost = OPENING, [], 0.0
    for d_, p_ in zip(demand, prod):
        s += p_ - d_
        if s < 0: lost += -s; s = 0.0          # no backorders: rivals hold ~900 KT of stock
        path.append(s)
    path = np.array(path); excess = max(path[-1] - FLOOR, 0)
    return {"Demand KT": demand.sum() / 1e3, "Supply KT": (prod.sum() + OPENING) / 1e3,
            "Lost sales KT": lost / 1e3, "Lost gross profit PKR bn": lost * PRICE_T * GM / 1e9,
            "Year-end excess KT": excess / 1e3, "Discount to clear PKR bn": excess * DISC_T / 1e9,
            "Avg stock KT": path.mean() / 1e3, "Avg working capital PKR bn": path.mean() * COST_T / 1e9,
            "Financing cost PKR m": path.mean() * COST_T * FIN_RATE / 1e6,
            "Weeks below floor": int((path < FLOOR).sum()), "_path": path}

rabi = (plan["Event_Flag"] == 1) & ((plan["Week_Start_Date"] >= "2026-10-01") | (plan["Week_Start_Date"] < "2026-03-01"))
scenarios = {
    "Base":                      (base,                                  production()),
    "Downside -10%":             (base * 0.9,                            production()),
    "Downside -20%":             (base * 0.8,                            production()),
    "Upside +10%":               (base * 1.1,                            production()),
    "Upside +20%":               (base * 1.2,                            production()),
    "Rabi peak 25% stronger":    (np.where(rabi, base * 1.25, base),     production()),
    "Rabi peak 25% weaker":      (np.where(rabi, base * 0.75, base),     production()),
    "Port Qasim gas -50% in Q4": (base, production(cuts=[("2026-09-28", 13, PQ_RATE * 0.5)])),
    # Flexible responses the decision allows, triggered from end-Q1 (rolling tracking signal < -4)
    "Downside -10% + trigger":   (base * 0.9, production(cuts=[("2026-03-30", 39, PQ_RATE * 0.5), ("2026-06-29", TA_WEEKS, TA_LOSS / TA_WEEKS)])),
    "Downside -20% + trigger":   (base * 0.8, production(cuts=[("2026-03-30", 39, PQ_RATE * 0.5), ("2026-06-29", TA_WEEKS, TA_LOSS / TA_WEEKS)])),
    "Upside +10% + defer turnaround": (base * 1.1, production(ta_start=None)),
    "Turnaround in Oct-Nov (2025 pattern), Base": (base, production(ta_start="2026-10-05")),
}
res = {k: simulate(d, p) for k, (d, p) in scenarios.items()}
stress = pd.DataFrame({k: {m: v for m, v in r.items() if m != "_path"} for k, r in res.items()}).T
pd.options.display.float_format = "{:,.1f}".format
try:
    display(stress)
except NameError:
    print(stress.to_string())

# ---------- Chart: stock path under each core scenario ----------
fig, ax = plt.subplots(figsize=(13, 6))
for k, c in [("Base", "#1F3864"), ("Downside -10%", "#E07B00"), ("Downside -20%", "#C0392B"),
             ("Upside +10%", "#2E9E44"), ("Upside +20%", "#117A65"), ("Downside -10% + trigger", "#E07B00")]:
    ax.plot(plan["Week_Start_Date"], res[k]["_path"] / 1000, color=c, lw=2.2, ls="--" if "trigger" in k else "-", label=k)
ax.axhline(FLOOR / 1000, color="grey", ls=":", lw=1.5, label=f"Safety floor {FLOOR/1000:.0f} KT")
ax.axvspan(pd.Timestamp("2026-02-23"), pd.Timestamp("2026-04-13"), color="#EAF0F8", label="Planned turnaround window")
ax.set_ylabel("Closing urea stock, KT"); ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8.5, ncol=2, loc="upper left")
ax.set_title("2026 stress test — FFC Sona Urea stock under demand and supply scenarios", loc="left", fontweight="bold")
fig.text(0.01, 0.005, LABEL, fontsize=7.5, color="#C00000", style="italic")
plt.tight_layout(rect=(0, 0.03, 1, 1)); plt.savefig("FFC_stress_test.png", dpi=180); plt.show()

## Audit corrections
Rolling-origin model comparison, the averaged forecast, the adjustable plan, and validation of the selected forecast.

In [ ]:
# %% [17] AUDIT: rolling-origin comparison over 17 windows (12 weeks each)
rows = []
for o in range(80, len(df) - 11, 4):
    trn, tst = df.iloc[:o], df.iloc[o:o + 12]
    a = tst["Actual_Demand"].to_numpy(float); y = trn["Actual_Demand"].to_numpy(float)
    dec, _, _ = fit_decomp_event(trn, tst)
    sn = df["Actual_Demand"].to_numpy(float)[o - 52:o - 40]          # same weeks one year earlier
    def ses(al):
        lv = y[0]
        for v in y[1:]: lv = al * v + (1 - al) * lv
        return np.full(12, lv)
    cands = {"Average (Decomp + Seasonal Naive)": (dec + sn) / 2, "Seasonal Decomp + Event": dec, "Seasonal Naive": sn,
             "SES (alpha 0.3)": ses(0.3), "SES (alpha 0.6)": ses(0.6), "Moving Average (4 wk)": np.full(12, y[-4:].mean()), "Naive": np.full(12, y[-1])}
    rows.append({k: np.abs(a - f).sum() / a.sum() * 100 for k, f in cands.items()})
rolling = pd.DataFrame(rows)
print(f"{len(rolling)} rolling windows. Mean WAPE (%):"); print(rolling.mean().round(1).sort_values().to_string())
print("Decomposition beats Seasonal Naive in", int((rolling["Seasonal Decomp + Event"] < rolling["Seasonal Naive"]).sum()), "windows")

In [ ]:
# %% [18] AUDIT: averaged 2026 forecast and the adjustable plan
DISC_T = 1_479 * 20                                   # discount cost at 2026 list price
last52 = df.iloc[-52:]
sn26 = last52["Actual_Demand"].to_numpy(float) / np.where(last52["Promo_Flag"] == 1, 1 + PROMO_LIFT, 1.0)
dec26 = fit_baseline(df, plan, PROMO_LIFT)
central = (dec26 + sn26) / 2
print(LABEL)
print(f"2026 demand, KT: low {sn26.sum()/1e3:,.0f} | central {central.sum()/1e3:,.0f} | high {dec26.sum()/1e3:,.0f}")

def adjustable_plan(demand):
    p = production()                                   # full rate, turnaround 23 Feb
    p[:13] -= PQ_RATE * 0.5                            # Port Qasim at 50% in Q1
    dev = demand[:13].sum() / central[:13].sum() - 1   # end-Q1 check against central forecast
    if dev > 0.075:   rate, act = 1.0, "Port Qasim 100% from April"
    elif dev < -0.075:
        rate, act = 0.0, "Port Qasim off from April + July maintenance"
        j = week_idx("2026-07-06"); p[j:j + TA_WEEKS] -= TA_LOSS / TA_WEEKS
    else:             rate, act = 0.5, "No change"
    p[13:] -= PQ_RATE * (1 - rate)
    return p, act

cases = {"Low: 2025 repeats": sn26, "Central": central, "High": dec26, "Central -10%": central * 0.9,
         "Central +10%": central * 1.1, "Central -20%": central * 0.8}
out = {}
for k, dmd in cases.items():
    p, act = adjustable_plan(dmd)
    new, old = simulate(dmd, p), simulate(dmd, production())
    out[k] = {"Trigger": act, "Lost KT": new["Lost sales KT"], "Excess KT": new["Year-end excess KT"],
              "Cost PKR bn": new["Lost gross profit PKR bn"] + new["Discount to clear PKR bn"],
              "Old full-rate plan cost PKR bn": old["Lost gross profit PKR bn"] + old["Discount to clear PKR bn"]}
summary = pd.DataFrame(out).T
try:
    display(summary)
except NameError:
    print(summary.to_string())
p, _ = adjustable_plan(central)
print(f"Next 12 weeks: produce {p[:12].sum()/1e3:,.0f} KT vs central demand {central[:12].sum()/1e3:,.0f} KT")

In [ ]:
# %% [19] AUDIT: holdout validation of the selected averaged forecast (Validation Note figures)
avg_fc = (forecasts["Seasonal Decomp + Event"] + forecasts["Seasonal Naive"]) / 2
err = y_test - avg_fc
print(LABEL)
print("Averaged forecast, holdout:", {k: round(v, 1) for k, v in metrics(y_test, avg_fc).items()})
ts_path = np.cumsum(err) / (np.cumsum(np.abs(err)) / np.arange(1, H + 1))
print("Tracking-signal path:", ts_path.round(1))
val_avg = test_df[["Week_Start_Date", "Actual_Demand", "Promo_Flag", "Promo_Depth", "Event_Flag", "Exceptional_Event"]].copy()
val_avg["Forecast"] = avg_fc.round(0); val_avg["Error (A-F)"] = err.round(0)
val_avg["Share of error %"] = (val_avg["Error (A-F)"].abs() / val_avg["Error (A-F)"].abs().sum() * 100).round(1)
top5_avg = val_avg.reindex(val_avg["Error (A-F)"].abs().sort_values(ascending=False).index).head(5)
try:
    display(top5_avg)
except NameError:
    print(top5_avg.to_string())
print("Top five share of holdout error:", round(top5_avg["Share of error %"].sum(), 1), "%")